# Suspension Velocity Analysis

This notebook analyzes suspension velocity distributions from shock pot displacement data. Understanding how your dampers work across different velocity ranges is crucial for proper setup.

## Interpretation Guide

| Pattern | Possible Cause | Potential Action |
|---------|----------------|------------------|
| High friction % | Damper friction, insufficient low-speed valving | Check damper seals, consider softer low-speed settings |
| Positive skew (more bump) | Nose dive, roll stiffness imbalance | Increase compression damping, check spring rates |
| Negative skew (more rebound) | Rear squat, extension issues | Increase rebound damping, check spring preload |
| High curb % | Aggressive kerb use, bottoming | Add bump stops, increase spring rate |
| Left/Right asymmetry | Weight distribution, alignment | Check corner weights, alignment, damper settings |
| Front/Rear asymmetry | Pitch balance issues | Adjust front/rear damping or spring ratio |

## What You'll Find Here

- **Velocity Histograms**: Distribution of wheel velocity for each corner (FL, FR, RL, RR)
- **Velocity Range Shading**: Visual indication of damper operating ranges
  - **Gray (Friction)**: < 5 mm/s - Static friction zone, damper may not move
  - **Light Blue (Slow)**: 5-25 mm/s - Low speed damping, affects body control
  - **Light Green (Fast)**: 25-200 mm/s - High speed damping, affects bump absorption
  - **Light Coral (Curb)**: > 200 mm/s - Curb/kerb strikes, bottoming
- **Statistics Tables**: Skew, symmetry, and corner comparisons

## Using Your Own Data

1. **Run the first cell** below to install packages and display the upload widget
2. **Click "Choose File"** to select your `.xrk`, `.xrz`, or `.ibt` file
3. **Configure channel names** if your shock pot channels have different names
4. **Run all remaining cells** to analyze your data

## Requirements

- Shock pot displacement channels (e.g., `LF_Shock_Pot`, `RF_Shock_Pot`, etc.)

**Note:** This notebook works in both JupyterLite (browser) and standard JupyterLab environments.

In [1]:
# Install required packages (needed for JupyterLite, skipped in regular JupyterLab if already installed)
%pip install -q motorsports-data-notebook

# Use the Rust parser backend for ~3x faster file loading
import os

os.environ["LIBXRK_BACKEND"] = "rust"

# Import helper functions
from motorsports_data_notebook.suspension import (
    MotionRatios,
    VelocityRanges,
    SUSPENSION_CHANNEL_NAMES,
    analyze_suspension_velocity,
    format_suspension_stats_table,
    format_symmetry_table,
    format_comparison_table,
)
from motorsports_data_notebook.visualization import (
    format_lap_time,
    plot_suspension_velocity_histogram,
    show_fig,
)
from motorsports_data_notebook.widgets import SessionPicker

# Session picker with channel configuration
# Upload your own file and select a lap to analyze
session = SessionPicker(
    default_file="../data/CMD_Inferno 86_Fuji GP Sh_Generic testing_a_2248.xrz",
    channel_mapping={
        "shock_fl": "LF_Shock_Pot",
        "shock_fr": "RF_Shock_Pot",
        "shock_rl": "LR_Shock_Pot",
        "shock_rr": "RR_Shock_Pot",
    },
)
session.display()

/home/runner/work/motorsports_data_notebook/motorsports_data_notebook/.venv/bin/python: No module named pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
# Get laps as pandas DataFrame for display
laps = session.get_laps()

In [3]:
# Display lap times table
laps.style.format({"lap_time": format_lap_time})  # type: ignore[dict-item]

,num,start_time,end_time,lap_type,lap_time
0,1,150454,279602,full,2:09.148
1,2,279602,406240,full,2:06.638
2,3,406240,532797,full,2:06.557
3,4,532797,659283,full,2:06.486
4,5,659283,787773,full,2:08.490
5,6,787773,913776,full,2:06.003
6,7,913776,1041398,full,2:07.622
7,8,1041398,1168323,full,2:06.925
8,9,1168323,1294676,full,2:06.353
9,10,1294676,1420573,full,2:05.897


## Configuration

Configure your shock pot channel names and motion ratios below. The defaults are set for a Toyota 86 ZN6 with standard AIM channel naming.

In [4]:
# Get the configured channel names
channel_names = session.get_channel_names()

# Motion ratios: wheel_velocity = shock_velocity / motion_ratio
# Default: Toyota 86 ZN6 (front MacPherson: 0.997, rear multi-link: 0.768)
motion_ratios = MotionRatios(
    front_left=0.997,
    front_right=0.997,
    rear_left=0.768,
    rear_right=0.768,
)

# Velocity range thresholds (mm/s) - defines the shading regions
velocity_ranges = VelocityRanges(
    friction=5.0,  # Below: static friction zone
    slow=25.0,  # Below: low speed damping
    fast=200.0,  # Above: curb/high speed
)

## Analyze Best Lap

In [5]:
# Get the selected lap
selected_lap = session.get_selected_lap()
log = session.get_log()
lap_num = int(selected_lap["num"])
print(f"Analyzing lap {lap_num} - {format_lap_time(selected_lap['lap_time'])}")

# Filter log to selected lap using libxrk 0.5.0 methods
lap_log = log.filter_by_lap(lap_num)

# Analyze suspension velocity (receives pre-filtered log)
result = analyze_suspension_velocity(
    lap_log,
    channel_names=channel_names,
    motion_ratios=motion_ratios,
    velocity_ranges=velocity_ranges,
    smoothing_window=5,  # Rolling average to reduce noise
    bin_size=10.0,  # Histogram bin size in mm/s
)

print("Analysis complete!")

Analyzing lap 12 - 2:05.388
Analysis complete!


## Velocity Histogram Visualization

This 4-quadrant plot shows the velocity distribution for each corner:
- **Blue bars**: Bump (compression) - positive velocity
- **Red bars**: Rebound (extension) - negative velocity
- **Background shading**: Velocity range zones

In [6]:
# Plot velocity histograms
fig = plot_suspension_velocity_histogram(
    result,
    title=f"Suspension Velocity Distribution - Lap {int(selected_lap['num'])} ({format_lap_time(selected_lap['lap_time'])})",
)
show_fig(fig)

## Statistics Tables

### Per-Corner Statistics

Key metrics for each corner:
- **Skew**: Distribution asymmetry (0 = symmetric, + = more bump, - = more rebound)
- **Kurtosis**: Distribution "tailedness" (0 = normal, + = heavy tails)
- **Mean/Std**: Average velocity and spread
- **Range %**: Percentage of time spent in each velocity range

In [7]:
# Per-corner statistics
stats_df = format_suspension_stats_table(result)
stats_df.style.format(
    {
        "Skew": "{:.2f}",
        "Kurtosis": "{:.2f}",
        "Mean (mm/s)": "{:.1f}",
        "Std (mm/s)": "{:.1f}",
        "Zero Bin %": "{:.1f}",
        "Friction %": "{:.1f}",
        "Slow Bump %": "{:.1f}",
        "Slow Rebound %": "{:.1f}",
        "Fast Bump %": "{:.1f}",
        "Fast Rebound %": "{:.1f}",
        "Curb %": "{:.1f}",
    }
)

,Corner,Skew,Kurtosis,Mean (mm/s),Std (mm/s),Zero Bin %,Friction %,Slow Bump %,Slow Rebound %,Fast Bump %,Fast Rebound %,Curb %
0,FL,1.85,555.70,0.0,28.4,25.4,25.4,28.0,32.9,6.5,7.2,0.1
1,FR,1.70,650.09,-0.0,27.5,28.2,28.2,27.0,32.8,5.9,5.8,0.2
2,RL,0.45,193.68,-0.0,48.5,18.9,18.9,24.0,22.5,16.9,17.1,0.5
3,RR,0.58,27.46,-0.0,48.2,21.6,21.6,22.3,24.1,15.4,15.7,0.9


### Bump vs Rebound Symmetry

Compare bump (compression) vs rebound (extension) for each corner:
- **Bump/Total %**: Percentage of non-friction time spent in bump (50% = symmetric)

In [8]:
# Bump vs rebound symmetry
symmetry_df = format_symmetry_table(result)
symmetry_df.style.format(
    {
        "Total Bump %": "{:.1f}",
        "Total Rebound %": "{:.1f}",
        "Bump/Total %": "{:.1f}",
        "Slow Bump %": "{:.1f}",
        "Slow Rebound %": "{:.1f}",
        "Fast Bump %": "{:.1f}",
        "Fast Rebound %": "{:.1f}",
    }
)

,Corner,Total Bump %,Total Rebound %,Bump/Total %,Slow Bump %,Slow Rebound %,Fast Bump %,Fast Rebound %
0,FL,34.4,40.1,46.2,28.0,32.9,6.5,7.2
1,FR,33.0,38.6,46.1,27.0,32.8,5.9,5.8
2,RL,40.9,39.7,50.8,24.0,22.5,16.9,17.1
3,RR,37.7,39.8,48.6,22.3,24.1,15.4,15.7


### Left vs Right and Front vs Rear Comparison

Compare combined wheel groups to identify balance issues:
- **Left vs Right**: Combines (FL + RL) vs (FR + RR) - shows lateral balance
- **Front vs Rear**: Combines (FL + FR) vs (RL + RR) - shows longitudinal balance

Positive values mean more time in that range for Left/Front, negative for Right/Rear.

In [9]:
# Left/Right and Front/Rear comparison
comparison_df = format_comparison_table(result)
comparison_df.style.format(
    {
        "Zero Bin Diff": "{:+.1f}",
        "Friction Diff": "{:+.1f}",
        "Slow Bump Diff": "{:+.1f}",
        "Slow Rebound Diff": "{:+.1f}",
        "Fast Bump Diff": "{:+.1f}",
        "Fast Rebound Diff": "{:+.1f}",
        "Curb Diff": "{:+.1f}",
    }
)

,Comparison,Zero Bin Diff,Friction Diff,Slow Bump Diff,Slow Rebound Diff,Fast Bump Diff,Fast Rebound Diff,Curb Diff
0,Left vs Right,-2.8,-2.8,+1.3,-0.8,+1.0,+1.4,-0.2
1,Front vs Rear,+6.6,+6.6,+4.4,+9.5,-10.0,-9.9,-0.6
